# 6 · Generation & Evaluation (Claude)

For each configuration: generate a grounded answer with Claude, score every retrieved chunk 0-5 with **Claude-as-a-judge**, and compute nDCG. Results are cached in `results/evaluated/`. **Machine 1** (API only, no local models).

In [ ]:
import json, time
from config import config
from indian_marriage_legal_recommender.generation import generate_answer
from indian_marriage_legal_recommender.evaluation import score_relevance, calculate_ndcg

config.EVALUATED_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
for key in config.PROFILES:
    contexts_path = config.CONTEXTS_DIR / f'{key}_contexts.json'
    rows = json.loads(contexts_path.read_text())
    evaluated = []
    for r in rows:
        t0 = time.perf_counter()
        ans = generate_answer(r['query'], r['contexts'])
        latency = time.perf_counter() - t0
        scores = score_relevance(r['query'], r['contexts'])
        evaluated.append({**r, 'answer': ans, 'relevance_scores': scores,
                          'ndcg': calculate_ndcg(scores), 'latency': latency})
    out = config.EVALUATED_DIR / f'{key}_final.json'
    out.write_text(json.dumps(evaluated, ensure_ascii=False, indent=2))
    print('Saved', out)

In [ ]:
# Export a flat Excel workbook (one sheet per config)
import json
import pandas as pd
from config import config

with pd.ExcelWriter(config.RESULTS_DIR / 'evaluation_results.xlsx') as writer:
    for key in config.PROFILES:
        data = json.loads((config.EVALUATED_DIR / f'{key}_final.json').read_text())
        recs = []
        for d in data:
            for j, ctx in enumerate(d['contexts']):
                recs.append({'ID': d['id'], 'Query': d['query'], 'Recommendation': ctx,
                             'Answer': d['answer'],
                             'Relevance': (d['relevance_scores'][j] if j < len(d['relevance_scores']) else None),
                             'nDCG': d['ndcg'], 'Latency': d['latency']})
        pd.DataFrame(recs).to_excel(writer, sheet_name=key[:31], index=False)
print('Saved evaluation_results.xlsx')